# Universidad Libre - Seccional Cali<br>Facultad de Ingeniería - Diplomado en Ciencia de Datos<br>(ↄ) Diego Fernando Marin, 2024

# 02_limpieza
Proyecto: **Identificación temprana del riesgo de deserción estudiantil en CEFIT**

---

Este cuaderno prepara los datos de CEFIT para el análisis, asegurando su calidad y consistencia.

**Propósito:** mejorar la calidad de los datos mediante corrección de errores, estandarización de formatos y reglas de calidad.

**Tareas que se realizan aquí:**
- Estandarización de nombres de columnas.
- Detección y tratamiento de valores nulos.
- Corrección de valores atípicos en la edad (edades negativas/imposibles).
- Homologación de categorías como `"Sin respuesta"` / `"SIN DATO"`.
- Conversión de tipos (fechas) y creación de variables nuevas.
- Construcción de la **variable objetivo**: *Riesgo de no continuidad*.

> **Nota sobre el origen de los datos:** CEFIT entregó la información como una única exportación institucional (un solo archivo), por lo que la etapa de *consolidación* (01) es trivial: basta con ubicar el archivo de CEFIT como `data/landing/datos_consolidados.csv`.

In [1]:
import os
import numpy as np
import pandas as pd

In [2]:
cwd = os.getcwd()  # current working directory
landing_dir = cwd + '/../data/landing/'
trusted_dir = cwd + '/../data/trusted/'

In [3]:
df = pd.read_csv(landing_dir + 'datos_consolidados.csv')
df.shape

(13461, 21)

In [4]:
df.head()

,Género,Municipio de Nacimiento,País,Municipio Dirección,Barrio Dirección,Estrato,Estado civil,EPS,Grupo Sisben,Zona,...,Ocupación,Medio de Transporte,Multiculturalidad,Estado,Fecha de Matrícula,Jornada,Programa,Período,Nivel,Edad
0,Femenino,Medellín,Colombia,Envigado,Las Flores,3.0,Soltero(a),EPS Sura,Sin respuesta,Urbana,...,Estudiante básica,A Pie,No aplica,Graduado,2018-02-05T10:00:00Z,MEDIA TÉCNICA,MEDIA TÉCNICA TECNICO LABORAL AUXILIAR ADMINIS...,2018-2,SEMESTRE 2,20.0
1,Femenino,Envigado,Colombia,Envigado,San José,2.0,Soltero(a),EPS Sura,Sin respuesta,Urbana,...,Estudiante superior,Público Metro,No aplica,Graduado,2016-01-19T10:00:00Z,NOCHE,10T-TÉCNICO LABORAL EN AUXILIAR ADMINISTRATIVO...,2016-2,SEMESTRE 2,29.0
2,Masculino,Bogotá,Colombia,La Mesa,Sin respuesta,3.0,Soltero(a),No Aplica,Sin respuesta,Rural,...,Estudiante superior,Transmilenio,No aplica,Cancelado,2021-02-08T10:00:00Z,NOCHE,09T-TÉCNICO LABORAL EN AUXILIAR ADMINISTRATIVO,2021-2,SEMESTRE 1,19.0
3,Masculino,Envigado,Colombia,Envigado,San José,2.0,Soltero(a),SISBEN,B3,Urbana,...,Estudiante superior,Público Bus o Buseta,No aplica,Graduado,2020-01-17T10:00:00Z,DIURNA,01T-TÉCNICO LABORAL AUXILIAR DE TALENTO HUMANO,2021-1,SEMESTRE 2,21.0
4,Masculino,Medellín,Colombia,Envigado,El Salado,2.0,Soltero(a),Nueva Promotora de Salud - Nueva EPS,N0,Urbana,...,Estudiante superior,Moto,No aplica,Inactivo,2020-01-27T10:00:00Z,NOCHE,27T-TÉCNICO LABORAL EN MERCADEO Y VENTAS (Estu...,2020-1,SEMESTRE 1,20.0


### 1. Estandarización de nombres de columnas

In [5]:
# Quitamos espacios sobrantes en los nombres de columna
df.columns = df.columns.str.strip()
list(df.columns)

['Género',
 'Municipio de Nacimiento',
 'País',
 'Municipio Dirección',
 'Barrio Dirección',
 'Estrato',
 'Estado civil',
 'EPS',
 'Grupo Sisben',
 'Zona',
 'Nivel de formación',
 'Ocupación',
 'Medio de Transporte',
 'Multiculturalidad',
 'Estado',
 'Fecha de Matrícula',
 'Jornada',
 'Programa',
 'Período',
 'Nivel',
 'Edad']

### 2. Tipos de datos y diagnóstico inicial

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 13461 entries, 0 to 13460
Data columns (total 21 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Género                   13461 non-null  str    
 1   Municipio de Nacimiento  13461 non-null  str    
 2   País                     13461 non-null  str    
 3   Municipio Dirección      13407 non-null  str    
 4   Barrio Dirección         13461 non-null  str    
 5   Estrato                  12193 non-null  float64
 6   Estado civil             13461 non-null  str    
 7   EPS                      13460 non-null  str    
 8   Grupo Sisben             13461 non-null  str    
 9   Zona                     12160 non-null  str    
 10  Nivel de formación       13461 non-null  str    
 11  Ocupación                11543 non-null  str    
 12  Medio de Transporte      13461 non-null  str    
 13  Multiculturalidad        13461 non-null  str    
 14  Estado                   13461 no

### 3. Valores nulos por columna

In [7]:
nulos = df.isnull().sum()
nulos = nulos[nulos > 0].sort_values(ascending=False)
print('Total de celdas vacías:', int(df.isnull().sum().sum()))
nulos

Total de celdas vacías: 6738


Ocupación              1918
Zona                   1301
Estrato                1268
Fecha de Matrícula     1217
Período                 366
Nivel                   366
Jornada                 123
Programa                123
Municipio Dirección      54
EPS                       1
Edad                      1
dtype: int64

### 4. Corrección de la variable Edad
La propuesta reportó edades imposibles (incluida una edad negativa). Marcamos como nulas las edades fuera de un rango razonable (`<= 0` o `> 100`).

In [8]:
df['Edad'] = pd.to_numeric(df['Edad'], errors='coerce')
print('Antes  -> min:', df['Edad'].min(), '| max:', df['Edad'].max())
df.loc[(df['Edad'] <= 0) | (df['Edad'] > 100), 'Edad'] = np.nan
print('Después -> min:', df['Edad'].min(), '| max:', df['Edad'].max())

Antes  -> min: -973.0 | max: 79.0
Después -> min: 1.0 | max: 79.0


### 5. Homologación de categorías de "no respuesta"
Unificamos `"Sin respuesta"`, `"SIN DATO"` y variantes en un único valor `"Sin dato"`, y limpiamos espacios en las columnas de texto.

In [9]:
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].replace({'Sin respuesta': 'Sin dato', 'SIN DATO': 'Sin dato',
                               'sin respuesta': 'Sin dato', 'No Aplica': 'No aplica',
                               'nan': np.nan})
df['EPS'].value_counts(dropna=False).head()

/tmp/ipykernel_588/2393014587.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include='object').columns:


EPS
EPS Sura           4989
No aplica          3050
SISBEN             1118
Coomeva EPS         699
Salud Total EPS     653
Name: count, dtype: int64

### 6. Conversión de fechas y variables nuevas
Convertimos la fecha de matrícula a tipo fecha y extraemos el **año del período** académico (el campo `Período` trae formatos como `2020-2`, `2020 - MT`).

In [10]:
df['Fecha de Matrícula'] = pd.to_datetime(df['Fecha de Matrícula'], errors='coerce')
df['Anio_Periodo'] = pd.to_numeric(df['Período'].str.extract(r'(\d{4})')[0], errors='coerce')
df[['Período', 'Anio_Periodo', 'Fecha de Matrícula']].head()

,Período,Anio_Periodo,Fecha de Matrícula
0,2018-2,2018.0,2018-02-05 10:00:00+00:00
1,2016-2,2016.0,2016-01-19 10:00:00+00:00
2,2021-2,2021.0,2021-02-08 10:00:00+00:00
3,2021-1,2021.0,2020-01-17 10:00:00+00:00
4,2020-1,2020.0,2020-01-27 10:00:00+00:00


### 7. Variable objetivo: *Riesgo de no continuidad*
Siguiendo la propuesta (y sujeto a validación con CEFIT), agrupamos el `Estado` en tres categorías.

In [11]:
NO_CONTINUIDAD = ['Desertor', 'Inactivo', 'Cancelado', 'Retiro - Aplazados']
CONTINUIDAD    = ['Activo', 'Graduado', 'Egresado']

def clasificar(estado):
    if estado in NO_CONTINUIDAD: return 'No continuidad'
    if estado in CONTINUIDAD:    return 'Continuidad'
    return 'Por validar'

df['Categoria'] = df['Estado'].apply(clasificar)
df['Categoria'].value_counts()

Categoria
Continuidad       6646
No continuidad    5677
Por validar       1138
Name: count, dtype: int64

**Último paso:** guardar los datos limpios, con calidad y estandarizados, en la capa `trusted`.

In [12]:
os.makedirs(trusted_dir, exist_ok=True)
df.to_csv(trusted_dir + 'datos_limpios.csv', index=False)
print('Guardado en data/trusted/datos_limpios.csv ->', df.shape)

Guardado en data/trusted/datos_limpios.csv -> (13461, 23)
